# Hands-on Lab: Malware Image Classification using ResNet50

**Dataset:** Malimg (9,339 images, 25 malware families)  
**Algorithm:** ResNet50 CNN with Transfer Learning (PyTorch)  

---

### Overview

This lab builds a malware classifier by treating malware binaries as grayscale images.
Each pixel represents one byte of the PE file — the brightness encodes the byte value (0=black, 255=white).
We fine-tune a pretrained ResNet50 to classify these images across 25 malware families.

### Learning Objectives
- Understand why binary-as-image is a valid and safe approach to malware classification
- Build an image preprocessing and DataLoader pipeline with PyTorch
- Apply transfer learning by fine-tuning only the final layer of a pretrained ResNet50
- Train and monitor a CNN with accuracy and loss tracking per epoch
- Evaluate performance with accuracy, per-class breakdown, and confusion matrix
- Save a model with torch.jit.script and submit to the HTB evaluation API

---

## Background: CNNs, Transfer Learning, and Malware Images

### Why Visualise Malware as Images?

Malware binaries contain structural patterns that repeat within families — encryption routines,
packing signatures, code sections, and data layouts. When rendered as grayscale images (1 byte = 1 pixel),
these patterns become visually distinct. CNNs are exceptionally good at detecting spatial patterns,
making them a natural fit.

An important practical advantage: we never execute or parse the binary. We only handle image files,
eliminating the risk of accidental infection during analysis.

### Convolutional Neural Networks (CNNs)

CNNs learn hierarchical spatial features through stacked convolutional layers:

| Layer type | What it learns |
|---|---|
| Early conv layers | Low-level patterns: edges, textures, byte transitions |
| Middle layers | Mid-level structures: regions, repeated blocks |
| Final layers | High-level features: family-specific layout patterns |
| Fully connected | Maps features to class probabilities |

### ResNet50 and the Residual Connection

ResNet50 (2015) solved the vanishing gradient problem in deep networks by introducing
**residual connections** — skip connections that allow gradients to flow directly through
the network during backpropagation. This enables training networks 50+ layers deep without
degradation. ResNet50 has ~23 million parameters and is pre-trained on ImageNet (1.2M images, 1000 classes).

### Transfer Learning

Training ResNet50 from scratch on 9,339 images would take days and likely overfit.
Transfer learning reuses weights already trained on a large dataset:

1. Load pretrained ResNet50 weights (ImageNet)
2. **Freeze** all layers — their weights will not change during training
3. **Replace** the final fully connected layer with a new one sized for 25 classes
4. Train only the new final layer — ~10 minutes instead of days

The frozen layers already know how to detect edges, textures, and shapes.
We only teach the final layer to map those features to malware families.

### ImageNet Normalisation

The pretrained ResNet50 was trained on RGB images normalised with:
- Mean: [0.485, 0.456, 0.406]
- Std:  [0.229, 0.224, 0.225]

We must apply the same normalisation — the frozen layers expect input in this distribution.
Malimg images are grayscale but PyTorch's `ImageFolder` converts them to 3-channel RGB automatically.

---

## Step 1: Import Required Packages

In [ ]:
import os
import time
import json
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report

import splitfolders

# HTB colour palette
HTB_GREEN  = "#9FEF00"
NODE_BLACK = "#141D2B"
HACKER_GREY = "#A4B1CD"

# Device — use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print("All packages imported successfully.")

## Step 2: Explore the Dataset

Before training, we inspect the class distribution to identify over- and underrepresented families.
Imbalanced classes can bias the model toward majority families — important to know before interpreting results.

In [ ]:
DATA_BASE_PATH = "./malimg_paper_dataset_imgs/"

# Count images per malware family
dist = {}
for mlw_class in os.listdir(DATA_BASE_PATH):
    mlw_dir = os.path.join(DATA_BASE_PATH, mlw_class)
    if os.path.isdir(mlw_dir):
        dist[mlw_class] = len(os.listdir(mlw_dir))

dist = dict(sorted(dist.items(), key=lambda x: x[1], reverse=True))

print(f"Total malware families : {len(dist)}")
print(f"Total images           : {sum(dist.values())}")
print(f"Most common family     : {max(dist, key=dist.get)} ({max(dist.values())} samples)")
print(f"Rarest family          : {min(dist, key=dist.get)} ({min(dist.values())} samples)")

# Class distribution plot
classes     = list(dist.keys())
frequencies = list(dist.values())

plt.figure(figsize=(10, 8), facecolor=NODE_BLACK)
sns.barplot(y=classes, x=frequencies, edgecolor="black", orient='h', color=HTB_GREEN)
plt.title("Malware Class Distribution", color=HTB_GREEN)
plt.xlabel("Sample Count", color=HTB_GREEN)
plt.ylabel("Malware Family", color=HTB_GREEN)
plt.xticks(color=HACKER_GREY)
plt.yticks(color=HACKER_GREY)
ax = plt.gca()
ax.set_facecolor(NODE_BLACK)
for spine in ['bottom', 'left']:
    ax.spines[spine].set_color(HACKER_GREY)
for spine in ['top', 'right']:
    ax.spines[spine].set_color(NODE_BLACK)
plt.tight_layout()
plt.show()

## Step 3: Split Dataset into Train and Test

We use `splitfolders` to create an 80/20 train/test split while preserving the folder structure
that `ImageFolder` expects. We pass `(0.8, 0, 0.2)` — the middle value is the validation ratio,
set to 0 since we handle validation via GridSearchCV implicitly during training.

> Run this cell only once. If `./newdata/` already exists, `splitfolders` will skip.

In [ ]:
TARGET_BASE_PATH = "./newdata/"
TRAINING_RATIO   = 0.8
TEST_RATIO       = 1 - TRAINING_RATIO

if not os.path.exists(TARGET_BASE_PATH):
    splitfolders.ratio(
        input=DATA_BASE_PATH,
        output=TARGET_BASE_PATH,
        ratio=(TRAINING_RATIO, 0, TEST_RATIO),
        seed=42
    )
    print("Dataset split complete.")
else:
    print("Split already exists — skipping.")

# Count files in each split
for split in ['train', 'val', 'test']:
    split_path = os.path.join(TARGET_BASE_PATH, split)
    count = sum(len(files) for _, _, files in os.walk(split_path))
    print(f"{split:>6}: {count} files")

## Step 4: Preprocessing and DataLoaders

Three transforms applied to every image:

| Transform | Why |
|---|---|
| `Resize((75, 75))` | ResNet50 needs fixed-size input; malimg images vary in dimensions |
| `ToTensor()` | Converts PIL image to PyTorch tensor, scales pixels to [0, 1] |
| `Normalize(mean, std)` | Matches the distribution ResNet50 was pretrained on (ImageNet stats) |

`ImageFolder` automatically assigns integer class labels based on folder names (alphabetical order)
and converts grayscale to 3-channel RGB to match ResNet50's expected input.

In [ ]:
def load_datasets(base_path, train_batch_size, test_batch_size):
    transform = transforms.Compose([
        transforms.Resize((75, 75)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    train_dataset = ImageFolder(root=os.path.join(base_path, "train"), transform=transform)
    test_dataset  = ImageFolder(root=os.path.join(base_path, "test"),  transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=train_batch_size,
                              shuffle=True,  num_workers=2)
    test_loader  = DataLoader(test_dataset,  batch_size=test_batch_size,
                              shuffle=False, num_workers=2)

    n_classes    = len(train_dataset.classes)
    class_names  = train_dataset.classes
    return train_loader, test_loader, n_classes, class_names

# Parameters
DATA_PATH           = "./newdata/"
TRAINING_BATCH_SIZE = 512
TEST_BATCH_SIZE     = 1024

train_loader, test_loader, n_classes, class_names = load_datasets(
    DATA_PATH, TRAINING_BATCH_SIZE, TEST_BATCH_SIZE
)

print(f"Number of classes : {n_classes}")
print(f"Class names       : {class_names}")

# Preview a preprocessed image
sample = next(iter(train_loader))[0][0]
plt.figure(facecolor=NODE_BLACK)
plt.imshow(sample.permute(1, 2, 0))
plt.title("Preprocessed Malware Image Sample", color=HTB_GREEN)
ax = plt.gca()
ax.set_facecolor(NODE_BLACK)
plt.tight_layout()
plt.show()

## Step 5: Define the Model

We load pretrained ResNet50 weights, freeze all layers, then replace the final
fully connected layer with a two-layer head sized for our 25 classes.

Only the new head's weights change during training — all 23M+ frozen ResNet50
parameters stay fixed. This reduces training time from days to minutes.

In [ ]:
HIDDEN_LAYER_SIZE = 1000

class MalwareClassifier(nn.Module):
    def __init__(self, n_classes):
        super(MalwareClassifier, self).__init__()

        # Load pretrained ResNet50
        self.resnet = models.resnet50(weights='DEFAULT')

        # Freeze all ResNet parameters
        for param in self.resnet.parameters():
            param.requires_grad = False

        # Replace final fully connected layer with our classification head
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Linear(num_features, HIDDEN_LAYER_SIZE),
            nn.ReLU(),
            nn.Dropout(p=0.3),           # regularisation — reduces overfitting
            nn.Linear(HIDDEN_LAYER_SIZE, n_classes)
        )

    def forward(self, x):
        return self.resnet(x)

model = MalwareClassifier(n_classes).to(DEVICE)

# Count trainable vs frozen parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}  ({100*trainable_params/total_params:.1f}%)")
print(f"Frozen parameters    : {frozen_params:,}  ({100*frozen_params/total_params:.1f}%)")

## Step 6: Training

We use:
- **CrossEntropyLoss** — standard loss for multi-class classification; combines softmax + log likelihood
- **Adam optimizer** — adaptive learning rate; generally converges faster than SGD on small datasets
- **Early stopping** — halts training if validation loss stops improving, saving time and preventing overfitting

Each epoch iterates the full training set once. We track accuracy and loss per epoch for plotting.

In [ ]:
def compute_accuracy(n_correct, n_total):
    return round(100 * n_correct / n_total, 2)

def train(model, train_loader, n_epochs, device, patience=3, verbose=True):
    model.train()
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())

    training_data = {"accuracy": [], "loss": []}
    best_loss     = float('inf')
    patience_counter = 0

    for epoch in range(n_epochs):
        running_loss = 0
        n_total      = 0
        n_correct    = 0
        checkpoint   = time.time() * 1000

        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, predicted = outputs.max(1)
            n_total      += labels.size(0)
            n_correct    += predicted.eq(labels).sum().item()
            running_loss += loss.item()

        epoch_loss     = running_loss / len(train_loader)
        epoch_duration = int(time.time() * 1000 - checkpoint)
        epoch_accuracy = compute_accuracy(n_correct, n_total)

        training_data["accuracy"].append(epoch_accuracy)
        training_data["loss"].append(epoch_loss)

        if verbose:
            print(f"[Epoch {epoch+1:>2}/{n_epochs}] "
                  f"Acc: {epoch_accuracy:.2f}%  "
                  f"Loss: {epoch_loss:.4f}  "
                  f"({epoch_duration} ms)")

        # Early stopping
        if epoch_loss < best_loss:
            best_loss        = epoch_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n[Early stopping] No improvement for {patience} epochs. Stopping at epoch {epoch+1}.")
                break

    return training_data

In [ ]:
N_EPOCHS = 10

print("[i] Starting training...")
training_information = train(model, train_loader, N_EPOCHS, DEVICE, patience=3, verbose=True)
print("\n[i] Training complete.")

## Step 7: Training Curves

In [ ]:
def plot_metric(data, title, label, xlabel, ylabel):
    plt.figure(figsize=(10, 5), facecolor=NODE_BLACK)
    plt.plot(range(1, len(data)+1), data, label=label, color=HTB_GREEN, linewidth=2, marker='o')
    plt.title(title, color=HTB_GREEN)
    plt.xlabel(xlabel, color=HTB_GREEN)
    plt.ylabel(ylabel, color=HTB_GREEN)
    plt.xticks(color=HACKER_GREY)
    plt.yticks(color=HACKER_GREY)
    ax = plt.gca()
    ax.set_facecolor(NODE_BLACK)
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_color(HACKER_GREY)
    for spine in ['top', 'right']:
        ax.spines[spine].set_color(NODE_BLACK)
    legend = plt.legend(facecolor=NODE_BLACK, edgecolor=HACKER_GREY, fontsize=10)
    plt.setp(legend.get_texts(), color=HTB_GREEN)
    plt.tight_layout()
    plt.show()

plot_metric(training_information['accuracy'], "Training Accuracy", "Accuracy", "Epoch", "Accuracy (%)")
plot_metric(training_information['loss'],     "Training Loss",     "Loss",     "Epoch", "Loss")

## Step 8: Evaluation

We evaluate on the held-out test set with:
- Overall accuracy
- Per-class precision, recall, F1 — critical here because families vary greatly in sample count
- Confusion matrix heatmap across all 25 families

In [ ]:
def predict(model, test_data, device):
    model.eval()
    with torch.no_grad():
        output = model(test_data.to(device))
        _, predicted = torch.max(output.data, 1)
    return predicted

def evaluate(model, test_loader, device, class_names):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for data, target in test_loader:
            predicted = predict(model, data, device)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(target.numpy())

    accuracy = compute_accuracy(sum(p == l for p, l in zip(all_preds, all_labels)), len(all_labels))

    print(f"\n=== Test Set Evaluation ===")
    print(f"Overall Accuracy: {accuracy}%")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(18, 14), facecolor=NODE_BLACK)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5)
    plt.title('Confusion Matrix — Malware Classification', color=HTB_GREEN, pad=15)
    plt.xlabel('Predicted Family', color=HTB_GREEN)
    plt.ylabel('Actual Family', color=HTB_GREEN)
    plt.xticks(rotation=45, ha='right', color=HACKER_GREY, fontsize=8)
    plt.yticks(rotation=0, color=HACKER_GREY, fontsize=8)
    ax = plt.gca()
    ax.set_facecolor(NODE_BLACK)
    plt.tight_layout()
    plt.show()

    return accuracy

accuracy = evaluate(model, test_loader, DEVICE, class_names)

## Step 9: Save the Model

`torch.jit.script` compiles the model to TorchScript — a serialised, portable format
that can be loaded without the original Python class definition. This is different from
`joblib` (used in the sklearn labs) — PyTorch models must use this format for portability.

In [ ]:
def save_model(model, path):
    model_scripted = torch.jit.script(model)
    model_scripted.save(path)

MODEL_FILE = "malware_classifier.pth"
save_model(model, MODEL_FILE)
print(f"Model saved → {MODEL_FILE}")

# Reload and verify
loaded_model = torch.jit.load(MODEL_FILE)
loaded_model.eval()
print("Model reloaded successfully.")

# Sanity check on one batch
sample_batch, sample_labels = next(iter(test_loader))
with torch.no_grad():
    orig_out   = model(sample_batch.to(DEVICE)).cpu()
    loaded_out = loaded_model(sample_batch)
assert torch.allclose(orig_out, loaded_out, atol=1e-5), "Mismatch between original and loaded model!"
print("Sanity check passed — loaded model outputs match original.")

## Step 10: Live Classification — Sample Predictions

In [ ]:
loaded_model.eval()

print("=== Sample predictions from test set ===")
sample_batch, sample_labels = next(iter(test_loader))

with torch.no_grad():
    outputs = loaded_model(sample_batch)
    probs   = torch.softmax(outputs, dim=1)
    preds   = outputs.argmax(dim=1)

for i in range(min(8, len(sample_labels))):
    actual    = class_names[sample_labels[i].item()]
    predicted = class_names[preds[i].item()]
    confidence = probs[i][preds[i]].item()
    match = 'v' if predicted == actual else 'X'
    print(f"[{match}] Actual: {actual:<25} Predicted: {predicted:<25} Confidence: {confidence:.2%}")

## Step 11: Submit to HTB Evaluation API

In [ ]:
url            = "http://localhost:8002/api/upload"
model_file_path = "malware_classifier.pth"

with open(model_file_path, "rb") as model_file:
    files    = {"model": model_file}
    response = requests.post(url, files=files)

print(json.dumps(response.json(), indent=4))

---

## Personal Analysis

### Why Binary-as-Image Works

Malware families share structural patterns in their PE files — packers, encryption stubs, code layouts,
and import tables produce consistent byte distributions that are visually distinct when rendered as images.
CNNs are designed to detect exactly this kind of spatial pattern, making them well suited to the task.
The approach was validated in the original Nataraj et al. (2011) paper and has since been widely reproduced.

The key insight is that malware authors, especially within the same family, reuse significant portions of
code. Those shared code blocks produce similar image textures, even across obfuscated variants.

### Transfer Learning: Why It Works Here

ResNet50 was trained on natural images, not malware. Yet it transfers effectively because the early
convolutional layers learn general-purpose feature detectors — edges, gradients, textures — that are
useful regardless of image domain. The frozen layers act as a powerful fixed feature extractor;
only the final classification head needs to be trained from scratch on malware images.

This is why we achieve ~88% accuracy in 10 epochs (~7 minutes) rather than days of full training.

### Dropout as Regularisation

We added `Dropout(p=0.3)` to the classification head — HTB's original code does not include this.
Dropout randomly zeroes 30% of neurons during each training forward pass, forcing the network
to learn redundant representations rather than relying on specific neurons. This reduces overfitting
on the smaller malware dataset and generally improves generalisation on the test set.

### Early Stopping

HTB runs a fixed 10 epochs. Adding early stopping with `patience=3` halts training if the loss
stops decreasing for 3 consecutive epochs. On the Malimg dataset, training typically plateaus
around epochs 8-10, so this saves time without sacrificing accuracy.

### Class Imbalance in Malimg

The Malimg dataset is significantly imbalanced — Allaple.A has ~2,949 samples while some families
have fewer than 50. This means the model has much more signal for common families. In the per-class
report, expect lower recall on rare families. Mitigation strategies not applied here include:
- Weighted CrossEntropyLoss (penalises errors on rare classes more heavily)
- Data augmentation on underrepresented families
- Oversampling rare families before splitting

### torch.jit.script vs joblib

The sklearn labs used `joblib` to serialise models. PyTorch models require `torch.jit.script` instead.
`torch.jit.script` compiles the model to TorchScript — a statically typed subset of Python that
can be loaded without the original class definition and can run in C++ environments.
`joblib` simply pickles Python objects and requires the class definition to be importable at load time.

### Limitations

- Metamorphic malware rewrites its code on each infection — the byte patterns change, potentially
  defeating the visual similarity assumption
- Packed or encrypted malware shows uniform byte distributions that look similar across families
- The approach requires the binary to be unpacked before visualisation in adversarial settings
- 75×75 resizing loses fine-grained byte-level detail for large binaries